# Single Fidelity KAN Spot Checks

Quick single-model sanity runs for `SurrogateModels.mfkans.KANModel`, split out of `SingleFidelityComparison_KAN.ipynb` so they can be iterated on in minutes instead of paying for a full multi-hour sweep every time a hyperparameter changes.

For each of the 2D/4D/6D/8D test functions, this runs **two** training calls on the same train/test split: one with no regularization (`lamb=0`, pykan's own default) and one with `lamb=1e-3`, so the effect of regularization on the train/test gap can be compared directly. All runs use `grid=[3, 5]`, `n=1000` training points, and `n_iter=400` steps per grid stage.

This grid schedule is deliberately more conservative than earlier experiments (`grid=[3, 5, 10]` or further) -- the `5 -> 10` refinement step was found to cause clear overfitting on the 4D case (train error kept dropping while test error rose and plateaued), consistent with adding far more spline-coefficient capacity than 1000 points in 4D can constrain. `lamb=1e-3` is a first attempt at countering that same failure mode through regularization instead of (or alongside) a more conservative grid.

Once a combination here looks good, move its settings into `SingleFidelityComparison_KAN.ipynb` for the full sample-size sweep.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import examples.multi_fidelity_functions as mfs
from examples import single_fidelity_kan_helpers as kan_helpers


## Important gotchas with `KANModel`

- `KANModel` does not cast its inputs -- the underlying `kan.KAN` network runs in `float32`, so `x_mat`/`y_vec` (and the test arrays) must already be `float32`, or training crashes with a dtype mismatch. `kan_helpers.create_test_points` and `kan_helpers.generate_kan_model(_comparisons)` handle this cast for you.
- `width` must start with the input dimension and end with the output dimension (`1` here), e.g. `[2, 5, 5, 1]` for the 2D case -- unlike `MLPNetwork`'s `hidden_layer_size`, which only specifies the hidden layers.
- `grid` is consumed (`grid.pop(0)`) as the model refines through each entry, so passing the same list object into multiple training runs would leave it empty after the first call. The helpers always hand `KANModel` a fresh copy, so you can safely reuse one `grid=[...]` default across calls.
- `scale_input`/`scale_output` default to `True` in these helpers (bounded `[-1, 1]` scaling on `x` using the function's known domain bounds, z-score scaling on `y`) -- `model.model_error()` and `.evaluate()` both work in original units regardless.
- `lamb` controls pykan's complexity regularization (`objective = train_loss + lamb * reg_`); it defaults to `0.0` (no regularization) unless passed explicitly.

### 2-Dimensional Dataset

In [ ]:
key = "2D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = kan_helpers.create_test_points(my_function, n=1000)


#### 2D -- no regularization (`lamb=0`)

In [ ]:
model_2d_nolamb, error_2d_nolamb, train_time_2d_nolamb, eval_time_2d_nolamb = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[5, 5],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=0.0,
)


In [ ]:
ax = model_2d_nolamb.plot_fit_error("2D KAN Fit Error n=1000, lamb=0")
plt.yscale("log")
kan_helpers.save_plot(ax, "2D_grid3-5_lamb0_width2-5-5-1")
plt.show()
print(f"2D (lamb=0) model_error = {error_2d_nolamb}")


#### 2D -- with regularization (`lamb=1e-3`)

In [ ]:
model_2d_lamb1e3, error_2d_lamb1e3, train_time_2d_lamb1e3, eval_time_2d_lamb1e3 = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[5, 5],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=1e-3,
)


In [ ]:
ax = model_2d_lamb1e3.plot_fit_error("2D KAN Fit Error n=1000, lamb=1e-3")
plt.yscale("log")
kan_helpers.save_plot(ax, "2D_grid3-5_lamb1e-3_width2-5-5-1")
plt.show()
print(f"2D (lamb=1e-3) model_error = {error_2d_lamb1e3}")


In [ ]:
print(f"2D lamb=0     -> model_error={error_2d_nolamb}, train_time={train_time_2d_nolamb:.1f}s")
print(f"2D lamb=1e-3  -> model_error={error_2d_lamb1e3}, train_time={train_time_2d_lamb1e3:.1f}s")

# Rough convergence-quality check (see kan_helpers.convergence_report docstring)
kan_helpers.print_convergence_report("2D lamb=0", model_2d_nolamb.train_losses, model_2d_nolamb.test_losses)
kan_helpers.print_convergence_report("2D lamb=1e-3", model_2d_lamb1e3.train_losses, model_2d_lamb1e3.test_losses)


### 4-Dimensional Dataset

In [ ]:
key = "4D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = kan_helpers.create_test_points(my_function, n=1000)


#### 4D -- no regularization (`lamb=0`)

In [ ]:
model_4d_nolamb, error_4d_nolamb, train_time_4d_nolamb, eval_time_4d_nolamb = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[8, 8],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=0.0,
)


In [ ]:
ax = model_4d_nolamb.plot_fit_error("4D KAN Fit Error n=1000, lamb=0")
plt.yscale("log")
kan_helpers.save_plot(ax, "4D_grid3-5_lamb0_width4-8-8-1")
plt.show()
print(f"4D (lamb=0) model_error = {error_4d_nolamb}")


#### 4D -- with regularization (`lamb=1e-3`)

In [ ]:
model_4d_lamb1e3, error_4d_lamb1e3, train_time_4d_lamb1e3, eval_time_4d_lamb1e3 = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[8, 8],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=1e-3,
)


In [ ]:
ax = model_4d_lamb1e3.plot_fit_error("4D KAN Fit Error n=1000, lamb=1e-3")
plt.yscale("log")
kan_helpers.save_plot(ax, "4D_grid3-5_lamb1e-3_width4-8-8-1")
plt.show()
print(f"4D (lamb=1e-3) model_error = {error_4d_lamb1e3}")


#### 4D -- with regularization (`lamb=1e-4`)

In [ ]:
model_4d_lamb1e4, error_4d_lamb1e4, train_time_4d_lamb1e4, eval_time_4d_lamb1e4 = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[8, 8],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=1e-4,
)


In [ ]:
ax = model_4d_lamb1e4.plot_fit_error("4D KAN Fit Error n=1000, lamb=1e-4")
plt.yscale("log")
kan_helpers.save_plot(ax, "4D_grid3-5_lamb1e-4_width4-8-8-1")
plt.show()
print(f"4D (lamb=1e-4) model_error = {error_4d_lamb1e4}")


In [ ]:
print(f"4D lamb=0     -> model_error={error_4d_nolamb}, train_time={train_time_4d_nolamb:.1f}s")
print(f"4D lamb=1e-3  -> model_error={error_4d_lamb1e3}, train_time={train_time_4d_lamb1e3:.1f}s")
print(f"4D lamb=1e-4  -> model_error={error_4d_lamb1e4}, train_time={train_time_4d_lamb1e4:.1f}s")

# Rough convergence-quality check (see kan_helpers.convergence_report docstring)
kan_helpers.print_convergence_report("4D lamb=0", model_4d_nolamb.train_losses, model_4d_nolamb.test_losses)
kan_helpers.print_convergence_report("4D lamb=1e-3", model_4d_lamb1e3.train_losses, model_4d_lamb1e3.test_losses)
kan_helpers.print_convergence_report("4D lamb=1e-4", model_4d_lamb1e4.train_losses, model_4d_lamb1e4.test_losses)


### 6-Dimensional Dataset

In [ ]:
key = "6D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = kan_helpers.create_test_points(my_function, n=1000)


#### 6D -- no regularization (`lamb=0`)

In [ ]:
model_6d_nolamb, error_6d_nolamb, train_time_6d_nolamb, eval_time_6d_nolamb = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[12, 12],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=0.0,
)


In [ ]:
ax = model_6d_nolamb.plot_fit_error("6D KAN Fit Error n=1000, lamb=0")
plt.yscale("log")
kan_helpers.save_plot(ax, "6D_grid3-5_lamb0_width6-12-12-1")
plt.show()
print(f"6D (lamb=0) model_error = {error_6d_nolamb}")


#### 6D -- with regularization (`lamb=1e-3`)

In [ ]:
model_6d_lamb1e3, error_6d_lamb1e3, train_time_6d_lamb1e3, eval_time_6d_lamb1e3 = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[12, 12],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=1e-3,
)


In [ ]:
ax = model_6d_lamb1e3.plot_fit_error("6D KAN Fit Error n=1000, lamb=1e-3")
plt.yscale("log")
kan_helpers.save_plot(ax, "6D_grid3-5_lamb1e-3_width6-12-12-1")
plt.show()
print(f"6D (lamb=1e-3) model_error = {error_6d_lamb1e3}")


In [ ]:
print(f"6D lamb=0     -> model_error={error_6d_nolamb}, train_time={train_time_6d_nolamb:.1f}s")
print(f"6D lamb=1e-3  -> model_error={error_6d_lamb1e3}, train_time={train_time_6d_lamb1e3:.1f}s")

# Rough convergence-quality check (see kan_helpers.convergence_report docstring)
kan_helpers.print_convergence_report("6D lamb=0", model_6d_nolamb.train_losses, model_6d_nolamb.test_losses)
kan_helpers.print_convergence_report("6D lamb=1e-3", model_6d_lamb1e3.train_losses, model_6d_lamb1e3.test_losses)


### 8-Dimensional Dataset

In [ ]:
key = "8D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = kan_helpers.create_test_points(my_function, n=1000)


#### 8D -- no regularization (`lamb=0`)

In [ ]:
model_8d_nolamb, error_8d_nolamb, train_time_8d_nolamb, eval_time_8d_nolamb = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[16, 16],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=0.0,
)


In [ ]:
ax = model_8d_nolamb.plot_fit_error("8D KAN Fit Error n=1000, lamb=0")
plt.yscale("log")
kan_helpers.save_plot(ax, "8D_grid3-5_lamb0_width8-16-16-1")
plt.show()
print(f"8D (lamb=0) model_error = {error_8d_nolamb}")


#### 8D -- with regularization (`lamb=1e-3`)

In [ ]:
model_8d_lamb1e3, error_8d_lamb1e3, train_time_8d_lamb1e3, eval_time_8d_lamb1e3 = kan_helpers.generate_kan_model(
    name=key,
    func=my_function,
    n=1000,
    seed=0,
    X_test=X_test,
    y_test=y_test,
    hidden_layers=[16, 16],
    grid=[3, 5],
    spline_order=3,
    opt="LBFGS",
    n_iter=400,
    lr=1.0,
    lamb=1e-3,
)


In [ ]:
ax = model_8d_lamb1e3.plot_fit_error("8D KAN Fit Error n=1000, lamb=1e-3")
plt.yscale("log")
kan_helpers.save_plot(ax, "8D_grid3-5_lamb1e-3_width8-16-16-1")
plt.show()
print(f"8D (lamb=1e-3) model_error = {error_8d_lamb1e3}")


In [ ]:
print(f"8D lamb=0     -> model_error={error_8d_nolamb}, train_time={train_time_8d_nolamb:.1f}s")
print(f"8D lamb=1e-3  -> model_error={error_8d_lamb1e3}, train_time={train_time_8d_lamb1e3:.1f}s")

# Rough convergence-quality check (see kan_helpers.convergence_report docstring)
kan_helpers.print_convergence_report("8D lamb=0", model_8d_nolamb.train_losses, model_8d_nolamb.test_losses)
kan_helpers.print_convergence_report("8D lamb=1e-3", model_8d_lamb1e3.train_losses, model_8d_lamb1e3.test_losses)
